# NB01 — Data Acquisition
**Project:** NAFLD Stage-Specific Biomarker Discovery (Single-Dataset Pipeline)  
**Dataset:** GSE135251 — Govaere et al., *Science Translational Medicine*, 2020  
**DOI:** [10.1126/scitranslmed.aba4448](https://doi.org/10.1126/scitranslmed.aba4448)

---

**Purpose:** Download the GSE135251 bulk RNA-seq dataset from NCBI GEO, extract raw
count data and sample metadata, convert Ensembl gene IDs to HGNC symbols, and save
clean CSV files for downstream analysis.

**Outputs:**
- `data/raw/expression_matrix_raw.csv` — Raw counts matrix (genes × samples), gene-symbol-indexed
- `data/metadata/metadata_raw.csv` — Parsed sample metadata (NAS score, fibrosis stage, etc.)


## 1. Install Dependencies

In [1]:
# ── DEPENDENCIES ──────────────────────────────────────────────────
# Install all required packages. This cell is safe to re-run.
import subprocess, sys

packages = [
    "GEOparse",
    "pandas",
    "numpy",
    "mygene",
    "requests",
]

for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"]
    )

print("All dependencies installed.")


All dependencies installed.


## 2. Configuration

In [2]:
# ── CONFIGURATION ─────────────────────────────────────────────────
import os, pathlib

RANDOM_SEED = 42
REDOWNLOAD  = True   # Set False to skip download and read from local files

# Paths (relative to this notebook's location in notebooks/)
PROJECT_ROOT = pathlib.Path("..").resolve()
DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_META    = PROJECT_ROOT / "data" / "metadata"

# Create directories
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_META.mkdir(parents=True, exist_ok=True)

# GEO accession
GEO_ACCESSION = "GSE135251"

# GEO FTP URLs (fallback if GEOparse fails)
SOFT_URL = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE135nnn/{GEO_ACCESSION}/soft/{GEO_ACCESSION}_family.soft.gz"
RAW_TAR_URL = f"https://www.ncbi.nlm.nih.gov/geo/download/?acc={GEO_ACCESSION}&format=file"

# Validation thresholds
MIN_SAMPLES = 200
MIN_GENES   = 15000

print(f"Project root : {PROJECT_ROOT}")
print(f"Data raw     : {DATA_RAW}")
print(f"Data metadata: {DATA_META}")
print(f"REDOWNLOAD   : {REDOWNLOAD}")


Project root : D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset
Data raw     : D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\raw
Data metadata: D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\metadata
REDOWNLOAD   : True


## 3. Imports

In [3]:
import numpy as np
import pandas as pd
import gzip
import tarfile
import io
import warnings
import requests
from pathlib import Path
from collections import OrderedDict

np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

print("All imports loaded.")


All imports loaded.


## 4. Download & Parse SOFT Metadata

Parse the GEO SOFT file to extract sample-level metadata.  
Each sample has 5 `characteristics_ch1` fields:
`nas score`, `fibrosis stage`, `group in paper`, `disease`, `Stage`.


In [4]:
soft_path = DATA_RAW / f"{GEO_ACCESSION}_family.soft.gz"

if REDOWNLOAD or not soft_path.exists():
    print(f"Downloading SOFT file for {GEO_ACCESSION}...")
    try:
        import GEOparse
        gse = GEOparse.get_GEO(
            geo=GEO_ACCESSION,
            destdir=str(DATA_RAW),
            silent=True
        )
        print(f"Downloaded via GEOparse: {soft_path}")
    except Exception as e:
        print(f"GEOparse failed ({e}), trying direct HTTP download...")
        r = requests.get(SOFT_URL, stream=True)
        r.raise_for_status()
        with open(soft_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded via HTTP: {soft_path}")
        import GEOparse
        gse = GEOparse.get_GEO(filepath=str(soft_path), silent=True)
else:
    print(f"Loading cached SOFT file: {soft_path}")
    import GEOparse
    gse = GEOparse.get_GEO(filepath=str(soft_path), silent=True)

print(f"Loaded GEO series: {gse.name}")
print(f"Number of samples (GSMs): {len(gse.gsms)}")


Downloaded via GEOparse: D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\raw\GSE135251_family.soft.gz
Loaded GEO series: GSE135251
Number of samples (GSMs): 216


## 5. Parse Sample Metadata

In [5]:
# Extract metadata from characteristics_ch1 fields
metadata_records = []

for gsm_name, gsm in gse.gsms.items():
    record = {"sample_id": gsm_name}

    # Parse characteristics_ch1
    chars = gsm.metadata.get("characteristics_ch1", [])
    for ch in chars:
        if ":" in ch:
            key, val = ch.split(":", 1)
            key = key.strip().lower()
            val = val.strip()
            record[key] = val

    metadata_records.append(record)

metadata_df = pd.DataFrame(metadata_records)
metadata_df = metadata_df.sort_values("sample_id").reset_index(drop=True)

# Convert numeric columns
for col in ["nas score", "fibrosis stage"]:
    if col in metadata_df.columns:
        metadata_df[col] = pd.to_numeric(metadata_df[col], errors="coerce")

print(f"Metadata shape: {metadata_df.shape}")
print(f"Columns: {metadata_df.columns.tolist()}")
print(f"\nDisease distribution:")
print(metadata_df["disease"].value_counts())
print(f"\nGroup in paper distribution:")
print(metadata_df["group in paper"].value_counts())
print(f"\nFibrosis stage distribution:")
print(metadata_df["fibrosis stage"].value_counts().sort_index())
print(f"\nNAS score range: {metadata_df['nas score'].min():.0f} – {metadata_df['nas score'].max():.0f}")
metadata_df.head()


Metadata shape: (216, 6)
Columns: ['sample_id', 'nas score', 'fibrosis stage', 'group in paper', 'disease', 'stage']

Disease distribution:
disease
NAFLD      206
Control     10
Name: count, dtype: int64

Group in paper distribution:
group in paper
NASH_F3       54
NASH_F2       53
NAFL          51
NASH_F0-F1    34
NASH_F4       14
control       10
Name: count, dtype: int64

Fibrosis stage distribution:
fibrosis stage
0    46
1    48
2    54
3    54
4    14
Name: count, dtype: int64

NAS score range: 0 – 8


,sample_id,nas score,fibrosis stage,group in paper,disease,stage
0,GSM3998167,4,2,NASH_F2,NAFLD,early
1,GSM3998168,7,3,NASH_F3,NAFLD,moderate
2,GSM3998169,7,2,NASH_F2,NAFLD,early
3,GSM3998170,6,3,NASH_F3,NAFLD,moderate
4,GSM3998171,5,0,NASH_F0-F1,NAFLD,early


## 6. Download & Combine Raw Count Files

GSE135251 provides per-sample raw RNA-seq count files (Ensembl gene IDs).  
We download the RAW.tar archive, extract all `.counts.txt.gz` files,
and combine them into a single (genes × samples) matrix.


In [6]:
raw_tar_path = DATA_RAW / f"{GEO_ACCESSION}_RAW.tar"
raw_dir = DATA_RAW / f"{GEO_ACCESSION}_RAW"

if REDOWNLOAD or not raw_tar_path.exists():
    print(f"Downloading RAW.tar ({GEO_ACCESSION})... this may take a few minutes.")
    r = requests.get(RAW_TAR_URL, stream=True)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    downloaded = 0
    with open(raw_tar_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
            downloaded += len(chunk)
    print(f"Downloaded: {raw_tar_path} ({downloaded / 1e6:.1f} MB)")
else:
    print(f"Using cached RAW.tar: {raw_tar_path}")

# Extract tar
if REDOWNLOAD or not raw_dir.exists() or len(list(raw_dir.glob("*.counts.txt.gz"))) == 0:
    raw_dir.mkdir(parents=True, exist_ok=True)
    print("Extracting RAW.tar...")
    with tarfile.open(raw_tar_path, "r") as tar:
        tar.extractall(path=raw_dir)
    print(f"Extracted to: {raw_dir}")

count_files = sorted(raw_dir.glob("*.counts.txt.gz"))
print(f"Found {len(count_files)} count files.")


Downloaded: D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\raw\GSE135251_RAW.tar (45.9 MB)
Extracting RAW.tar...


C:\Users\karim\AppData\Local\Temp\ipykernel_20380\1245078763.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=raw_dir)


Extracted to: D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\raw\GSE135251_RAW
Found 216 count files.


## 7. Build Expression Matrix

In [7]:
# Map GSM accessions to count files
# Filename format: GSM3998167_017-Ann-Daly_S1.counts.txt.gz
gsm_to_file = {}
for fp in count_files:
    gsm_id = fp.name.split("_")[0]
    gsm_to_file[gsm_id] = fp

print(f"Mapped {len(gsm_to_file)} GSM IDs to count files.")

# Read all count files and combine
# Each file: two columns (Ensembl_ID \t count), no header
print("Reading and combining count files...")
counts_dict = OrderedDict()
gene_ids = None

for i, (gsm_id, fpath) in enumerate(sorted(gsm_to_file.items())):
    with gzip.open(fpath, "rt") as f:
        lines = f.readlines()

    genes = []
    values = []
    for line in lines:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            genes.append(parts[0])
            values.append(int(parts[1]))

    if gene_ids is None:
        gene_ids = genes
    else:
        assert genes == gene_ids, f"Gene order mismatch in {gsm_id}"

    counts_dict[gsm_id] = values

    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(gsm_to_file)} samples...")

# Build DataFrame (genes × samples)
expression_raw = pd.DataFrame(
    counts_dict,
    index=gene_ids
)
expression_raw.index.name = "ensembl_id"

print(f"\nRaw expression matrix shape: {expression_raw.shape}")
print(f"  Genes (Ensembl IDs): {expression_raw.shape[0]}")
print(f"  Samples: {expression_raw.shape[1]}")
print(f"  Total counts range: {expression_raw.values.min()} – {expression_raw.values.max()}")
print(f"  Sparsity (zeros): {(expression_raw.values == 0).mean():.1%}")


Mapped 216 GSM IDs to count files.
Reading and combining count files...
  Processed 50/216 samples...
  Processed 100/216 samples...
  Processed 150/216 samples...
  Processed 200/216 samples...

Raw expression matrix shape: (64258, 216)
  Genes (Ensembl IDs): 64258
  Samples: 216
  Total counts range: 0 – 45760587
  Sparsity (zeros): 65.2%


## 8. Map Ensembl Gene IDs to HGNC Symbols

Use the MyGene.info API to convert Ensembl gene IDs (ENSG...) to human-readable
HGNC gene symbols. Genes without a valid symbol mapping are dropped.


In [8]:
import mygene
mg = mygene.MyGeneInfo()

ensembl_ids = expression_raw.index.tolist()
print(f"Querying MyGene.info for {len(ensembl_ids)} Ensembl IDs...")

# Query in batches (mygene handles batching internally)
results = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    returnall=True,
    verbose=False
)

# Build mapping dict
ensembl_to_symbol = {}
for hit in results["out"]:
    eid = hit.get("query", "")
    symbol = hit.get("symbol", None)
    if symbol and isinstance(symbol, str):
        ensembl_to_symbol[eid] = symbol

mapped = len(ensembl_to_symbol)
total = len(ensembl_ids)
print(f"Mapped: {mapped}/{total} ({mapped/total:.1%})")
print(f"Unmapped: {total - mapped}")

# Save mapping for reproducibility
mapping_df = pd.DataFrame(
    list(ensembl_to_symbol.items()),
    columns=["ensembl_id", "gene_symbol"]
)
mapping_path = DATA_META / "ensembl_to_symbol_mapping.csv"
mapping_df.to_csv(mapping_path, index=False)
print(f"Saved mapping: {mapping_path}")


Querying MyGene.info for 64258 Ensembl IDs...
Mapped: 46830/64258 (72.9%)
Unmapped: 17428
Saved mapping: D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\metadata\ensembl_to_symbol_mapping.csv


## 9. Apply Symbol Mapping & Deduplicate

In [9]:
# Filter to mapped genes only
mapped_mask = expression_raw.index.isin(ensembl_to_symbol)
expr_mapped = expression_raw.loc[mapped_mask].copy()

# Replace index with gene symbols
expr_mapped.index = expr_mapped.index.map(ensembl_to_symbol)
expr_mapped.index.name = "gene_symbol"

# Handle duplicate gene symbols: keep the one with highest mean expression
pre_dedup = expr_mapped.shape[0]
expr_mapped["_mean"] = expr_mapped.mean(axis=1)
expr_mapped = expr_mapped.sort_values("_mean", ascending=False)
expr_mapped = expr_mapped[~expr_mapped.index.duplicated(keep="first")]
expr_mapped = expr_mapped.drop(columns=["_mean"])
expr_mapped = expr_mapped.sort_index()

print(f"Before deduplication: {pre_dedup} genes")
print(f"After deduplication:  {expr_mapped.shape[0]} genes")
print(f"Samples: {expr_mapped.shape[1]}")


Before deduplication: 46830 genes
After deduplication:  42584 genes
Samples: 216


## 10. Save Outputs

In [10]:
# Save expression matrix (genes × samples)
expr_path = DATA_RAW / "expression_matrix_raw.csv"
expr_mapped.to_csv(expr_path)
print(f"Saved: {expr_path}")
print(f"  Shape: {expr_mapped.shape} (genes × samples)")

# Save metadata
meta_path = DATA_META / "metadata_raw.csv"
metadata_df.to_csv(meta_path, index=False)
print(f"Saved: {meta_path}")
print(f"  Shape: {metadata_df.shape}")


Saved: D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\raw\expression_matrix_raw.csv
  Shape: (42584, 216) (genes × samples)
Saved: D:\NAFLD-Biomarker-Discovery-V2\nafld-single-dataset\data\metadata\metadata_raw.csv
  Shape: (216, 6)


## 11. Validation

In [11]:
# ── VALIDATION ASSERTIONS ─────────────────────────────────────────
n_samples = expr_mapped.shape[1]
n_genes   = expr_mapped.shape[0]

print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"  Total samples   : {n_samples}")
print(f"  Total genes      : {n_genes}")
print(f"  Metadata rows    : {len(metadata_df)}")
print(f"  Metadata columns : {metadata_df.columns.tolist()}")
print()

# Check sample alignment
expr_samples = set(expr_mapped.columns)
meta_samples = set(metadata_df["sample_id"])
overlap = expr_samples & meta_samples
missing_in_expr = meta_samples - expr_samples
missing_in_meta = expr_samples - meta_samples

print(f"  Samples in both  : {len(overlap)}")
print(f"  In metadata only : {len(missing_in_expr)}")
print(f"  In expression only: {len(missing_in_meta)}")
print()

# Metadata completeness
for col in metadata_df.columns:
    n_missing = metadata_df[col].isna().sum()
    print(f"  {col:20s}: {n_missing} missing ({n_missing/len(metadata_df):.1%})")
print()

# Assertions
assert n_samples >= MIN_SAMPLES, f"FAIL: Only {n_samples} samples (need >= {MIN_SAMPLES})"
print(f"  [OK] Sample count >= {MIN_SAMPLES}")

assert n_genes >= MIN_GENES, f"FAIL: Only {n_genes} genes (need >= {MIN_GENES})"
print(f"  [OK] Gene count >= {MIN_GENES}")

assert len(overlap) == n_samples, f"FAIL: Sample mismatch between expression and metadata"
print(f"  [OK] All expression samples have metadata")

print()
print("ALL VALIDATIONS PASSED")
print("=" * 60)


VALIDATION SUMMARY
  Total samples   : 216
  Total genes      : 42584
  Metadata rows    : 216
  Metadata columns : ['sample_id', 'nas score', 'fibrosis stage', 'group in paper', 'disease', 'stage']

  Samples in both  : 216
  In metadata only : 0
  In expression only: 0

  sample_id           : 0 missing (0.0%)
  nas score           : 0 missing (0.0%)
  fibrosis stage      : 0 missing (0.0%)
  group in paper      : 0 missing (0.0%)
  disease             : 0 missing (0.0%)
  stage               : 0 missing (0.0%)

  [OK] Sample count >= 200
  [OK] Gene count >= 15000
  [OK] All expression samples have metadata

ALL VALIDATIONS PASSED


## 12. Output Manifest

In [12]:
# ── OUTPUT MANIFEST ───────────────────────────────────────────────
output_files = [
    DATA_RAW / "expression_matrix_raw.csv",
    DATA_META / "metadata_raw.csv",
    DATA_META / "ensembl_to_symbol_mapping.csv",
]

print("=" * 60)
print("OUTPUT MANIFEST")
print("=" * 60)
for fp in output_files:
    if fp.exists():
        size_mb = fp.stat().st_size / 1e6
        print(f"  [OK] {fp.name:40s} {size_mb:8.2f} MB")
    else:
        print(f"  [FAIL] {fp.name:40s} MISSING")
print("=" * 60)


OUTPUT MANIFEST
  [OK] expression_matrix_raw.csv                   24.59 MB
  [OK] metadata_raw.csv                             0.01 MB
  [OK] ensembl_to_symbol_mapping.csv                1.17 MB


## Session Summary

**Notebook:** NB01 — Data Acquisition  
**Dataset:** GSE135251 (Govaere et al., 2020)

### What Was Accomplished
- Downloaded GSE135251 SOFT file and parsed sample metadata
- Downloaded and extracted 216 per-sample raw RNA-seq count files
- Combined individual counts into a unified expression matrix
- Mapped Ensembl gene IDs to HGNC gene symbols via MyGene.info
- Deduplicated genes (kept highest mean expression per symbol)
- Saved clean expression matrix and metadata files

### Key Numbers
- Total samples: 216 (206 NAFLD + 10 controls)
- Total genes (Ensembl): ~60,000 → after symbol mapping and dedup: ~20,000+
- Data format: raw integer counts (RNA-seq)
- Metadata fields: nas score, fibrosis stage, group in paper, disease, stage

### Outputs Saved
- `data/raw/expression_matrix_raw.csv` — Raw counts, genes × samples
- `data/metadata/metadata_raw.csv` — Parsed metadata
- `data/metadata/ensembl_to_symbol_mapping.csv` — Ensembl → Symbol mapping

### Issues / Warnings
- Individual ballooning/inflammation scores not available in GEO metadata
- NAS score is composite only; labeling in NB02 will use NAS + fibrosis stage

### Next Notebook Requires
- `data/raw/expression_matrix_raw.csv`
- `data/metadata/metadata_raw.csv`
